In [1]:
from IPython.display import display 

import base64
import json
import requests
from datetime import datetime, timedelta
from tqdm import tqdm

import pandas as pd 
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
# Set pandas option to format floats as percentages with 2 decimal places
#pd.set_option('display.float_format', '{:.2}'.format)

import os 
import shutil
import numpy as np
import ast

import missingno as msno
import matplotlib.pyplot as plt


In [2]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules
from config import gam_info

from security_config import emplifi_key
from functions import lookup_loader, calculate_rolling_avg_country_split, gnl_expander
import test_functions
import functions 

In [3]:
platformID = 'TTK'

lookup = lookup_loader(gam_info, platformID, '3',
                       with_country=True, country_col=['TikTok Codes'],
                       with_pop_col=True)
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']

# service
cols = ['ServiceID', 'Lumen']
service_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", 
                              sheet_name='ServiceID', usecols=cols, keep_default_na=False )
test_functions.test_lookup_files(service_codes, ['ServiceID', 'Lumen'], [f"{platformID}_3_09", f"{platformID}_3_10", f"{platformID}_3_11"], 
                                 test_step = "lookup files - ensuring service is correct")


✅ Test TTK_3_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TTK_3_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TTK_3_02 passed: No empty values in lookup.
...updating logbook...

✅ Test TTK_3_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TTK_3_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TTK_3_05 passed: No empty values in lookup.
...updating logbook...

✅ Test TTK_3_06 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TTK_3_07 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TTK_3_08 passed: No empty values in lookup.
...updating logbook...

✅ Test TTK_3_09 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TTK_3_10 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TTK_3_11 passed: No empty values in lookup.
...updating logbook...



In [4]:
socialmedia_accounts.sample()

,PlatformID,Status,Comment,Channel ID,Channel Name,Service,ServiceID,Excluding UK,Channel URL,Channel Username,Linked FB Account,Start,End
359,TTK,active,52 weeks country,TTK746ce2b5-197a-4eb6-86d2-aa2a39a021f7,BBC Learning English,Learning English,ELT,NaN,NaN,NaN,143048895744759,2020-04-01,NaT


# read in 

In [5]:
cols_that_must_not_be_empty = ['author', 'insights_viewers_by_country',
                               'insights_avg_time_watched', 'duration', 
                               'insights_reach', 'insights_completion_rate']
dataframes = []
storage_dir = f"../data/raw/{platformID}/post_level/"
csv_files = [f for f in os.listdir(storage_dir) if f.endswith(".csv")]
for f in csv_files:
    file_path = os.path.join(storage_dir, f)
    try:
        parts = f.replace(".csv", "").split("_")
        file_timeinfo = parts[0]
        if platformID != parts[1]:
            print('something is wrong with platformID in filename!')
        platformID = parts[1]
        profile_id = platformID+parts[2]
        week_str = parts[3]

        df = pd.read_csv(file_path)
        # test columns that must not be nan here -> move them to issues folder
        #Detect any required column that is entirely NaN
        entirely_nan_mask = df[cols_that_must_not_be_empty].isna().all(axis=0)
        entirely_nan_cols = entirely_nan_mask[entirely_nan_mask].index.tolist()
        
        if len(entirely_nan_cols) > 0:
            issues_dir = f"../data/raw/{platformID}/post_level/issues"
            os.makedirs(issues_dir, exist_ok=True)
                
            dest_path = os.path.join(issues_dir, f)
            shutil.move(file_path, dest_path)
            print(f"⚠️ Moved to issues (entirely NaN cols {entirely_nan_cols}): {f}")
            continue
            
        df["platformID"] = platformID
        df["profile_id"] = profile_id
        df["w/c"] = pd.to_datetime(week_str)
        
        if not df.empty:
            dataframes.append(df)
    except pd.errors.EmptyDataError:
        print(f"❌ Could not read file (empty or malformed): {f}")

# Combine all non-empty DataFrames
if dataframes:
    post_level_df = pd.concat(dataframes, ignore_index=True)
    print("✅ Combined DataFrame created.")
    display(post_level_df.head())
else:
    print("🚫 No valid data found to combine.")


✅ Combined DataFrame created.


,attachments,author,authorId,content_type,created_time,duration,id,link,media,message,...,insights_likes,insights_reach,insights_reach_engagement_rate,insights_shares,insights_video_views,insights_view_time,insights_viewers_by_country,platformID,profile_id,w/c
0,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T19:00:00+00:00,51,7536981698806811926,https://www.tiktok.com/@bbcnews/video/75369816...,video,Japan’s prime minister has described the count...,...,47395,499763,9.90,662,604220,7914601,"[{'country': 'AU', 'percentage': 0.032}, {'cou...",TTK,TTK3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
1,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T17:00:00+00:00,14,7536980939285499158,https://www.tiktok.com/@bbcnews/video/75369809...,video,A meteorite that crashed into a home in the US...,...,1043913,9793095,10.99,24274,11865744,115027940,"[{'country': 'AU', 'percentage': 0.019}, {'cou...",TTK,TTK3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
2,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T15:15:37+00:00,251,7536975832674372886,https://www.tiktok.com/@bbcnews/video/75369758...,video,Are planes actually crashing more often? #Plan...,...,51695,509953,10.63,1854,595410,9325982,"[{'country': 'NL', 'percentage': 0.012}, {'cou...",TTK,TTK3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
3,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T14:32:50+00:00,18,7536964795443039510,https://www.tiktok.com/@bbcnews/video/75369647...,video,Israeli Prime Minister Benjamin Netanyahu says...,...,83979,2522087,4.05,4363,3233846,45208070,"[{'country': 'NG', 'percentage': 0.031}, {'cou...",TTK,TTK3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04
4,[{'image_url': 'https://p16-pu-sign-no.tiktokc...,"{'id': '3d596e6a-d239-434e-bb79-93e5d291b216',...",3d596e6a-d239-434e-bb79-93e5d291b216,post,2025-08-10T13:00:00+00:00,67,7536900507957349654,https://www.tiktok.com/@bbcnews/video/75369005...,video,"Zara, Next and Marks & Spencer have all had ad...",...,70995,591508,12.46,1725,785423,9696699,"[{'country': 'GB', 'percentage': 0.56}, {'coun...",TTK,TTK3d596e6a-d239-434e-bb79-93e5d291b216,2025-08-04


In [6]:
post_level_df = post_level_df.rename(columns={'platformID': 'PlatformID'})
# Count unique weeks per profile
week_counts = post_level_df.groupby('profile_id')['w/c'].nunique().reset_index()
week_counts.columns = ['profile_id', 'number_of_weeks']

# Optional: sort by number of weeks
week_counts = week_counts.sort_values(by='number_of_weeks', ascending=False)

# Display 
print(week_counts)

# Assuming post_level_df is already loaded and contains 'profile_id' and 'w/c'
# Create a pivot table: profiles as rows, weeks as columns
pivot_df = post_level_df.pivot_table(columns='profile_id', index='w/c', aggfunc='size')


                                 profile_id  number_of_weeks
0   TTK057b1686-d9f3-51fe-a129-5732678e609e               42
2   TTK2ac83179-5aa1-4b36-9a7c-a8418f72a799               42
4   TTK34e41ced-576d-430a-b590-55d0c1d241b1               42
7   TTK4be46cb0-a590-5e23-b3cf-ea59dd02c530               42
6   TTK3d596e6a-d239-434e-bb79-93e5d291b216               42
8   TTK54ac56ff-37ee-597e-89ec-d63de04c9df1               42
14  TTKc02ca653-c3b6-4b34-b210-711e12f9eb2d               42
12  TTKa53907d1-2b3d-57a1-be38-cea14a04dc0f               42
9   TTK746ce2b5-197a-4eb6-86d2-aa2a39a021f7               42
15  TTKc3390a5e-42f2-4652-99ff-b903b61d8fc7               42
16  TTKfbd019f7-9dcb-5f22-a0e0-86cfb49a5959               42
13  TTKa824bd24-fa59-4039-88bf-4b152ffa2881               42
1   TTK1ea28c7a-b8c5-4e98-9c8b-456832eb4f21               31
5   TTK3c4d1098-0742-41ed-9182-f7ea05f398cd               30
3   TTK2bf05522-b1ed-5751-a294-22f1d95f6cd3               26
10  TTK80244afe-5625-509

In [7]:

def extract_author_info(row):
    if pd.isna(row):
        return pd.Series({'id': None, 'name': None, 'url': None})

    if isinstance(row, str):
        try:
            author_dict = ast.literal_eval(row)
        except (ValueError, SyntaxError):
            return pd.Series({'id': None, 'name': None, 'url': None})
    elif isinstance(row, dict):
        author_dict = row
    else:
        return pd.Series({'id': None, 'name': None, 'url': None})

    return pd.Series({
        'id': platformID+author_dict.get('id'),
        'name': author_dict.get('name'),
        'url': author_dict.get('url')
    })

# Apply the function
post_level_df[["Channel ID", "Channel Name", "Channel URL"]] = post_level_df['author'].apply(extract_author_info)

In [8]:
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()

### RUN TESTS
# missing page_ids
test_functions.test_filter_elements_returned(post_level_df, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_3_12",
                                             "Check that all page IDs are found in SQL")


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=post_level_df,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_3_13",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
metrics_to_test = ['insights_avg_time_watched', 'duration', 'insights_reach',
                   'insights_completion_rate']
test_functions.test_non_null_and_positive(post_level_df, 
                           numeric_columns=metrics_to_test, 
                           test_number=f"{platformID}_3_14",
                           test_step='Check no missing values in page fans column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(post_level_df, ['Channel ID', 'w/c', 'link'], 
                               test_number=f"{platformID}_3_15",
                               test_step='Check no duplicates from redshift returned')

...testing Channel ID...
✅ Test TTK_3_12 passed: everything found!
...updating logbook...

❌ Missing weeks from Start onward:
                                  Channel ID      Start End        w/c
88   TTK1ea28c7a-b8c5-4e98-9c8b-456832eb4f21 2025-06-02 NaT 2025-06-30
94   TTK1ea28c7a-b8c5-4e98-9c8b-456832eb4f21 2025-06-02 NaT 2025-08-11
97   TTK1ea28c7a-b8c5-4e98-9c8b-456832eb4f21 2025-06-02 NaT 2025-09-01
101  TTK1ea28c7a-b8c5-4e98-9c8b-456832eb4f21 2025-06-02 NaT 2025-09-29
115  TTK1ea28c7a-b8c5-4e98-9c8b-456832eb4f21 2025-06-02 NaT 2026-01-05
116  TTK1ea28c7a-b8c5-4e98-9c8b-456832eb4f21 2025-06-02 NaT 2026-01-12

Summary of missing weeks per channel_id_col:
                                Channel ID  missing_week_count
0  TTK1ea28c7a-b8c5-4e98-9c8b-456832eb4f21                   6
...updating logbook...

✅ Test TTK_3_14 passed: No NaN and all numeric values > 0.
...updating logbook...

✅ Test TTK_3_15 passed: No combinations occurs more than once a week.
...updating logbook...



# Views

In [9]:

minnie_cols_used = {'Date': 'w/c', #minnie has a day by day breakdown and then calculates the average
               'Profile ID': "Channel ID", # author['name'],
               'Profile name': "Channel Name", # author['url'],
               'Profile URL': "Channel URL", #author['id'],
               #'Post detail URL': 'link',
               #'Content ID': 'link', # splice out from link
               'Platform': 'PlatformID', 
               'Content type': 'content_type',
               'Media type': 'media',
               #'Title': '', # missing
               #'Description': '', # missing
               'Content': 'message',
               #'Link URL': '', #unclear
               'View on platform': 'link',
               'Engagements': 'insights_engagements',
               'Total reach': 'insights_reach', #but number is different? 
               'Video length (sec)': 'duration',
               'Video view count': 'insights_video_views',
               'Total video view time (sec)': 'insights_view_time',
               'Average time watched (sec)': 'insights_avg_time_watched',
               'Completion rate': 'insights_completion_rate',
              }

views_df = post_level_df[minnie_cols_used.values()]
views_df['link'] = views_df['link'].fillna('').astype(str)
views_df['content_id'] = views_df['link'].str.split('/').str[-1].str.split('?').str[0]
views_df.head()

/var/folders/nk/x7c25nz92c1d6ljxz7l891cm0000gn/T/ipykernel_12460/3751647070.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  views_df['link'] = views_df['link'].fillna('').astype(str)
/var/folders/nk/x7c25nz92c1d6ljxz7l891cm0000gn/T/ipykernel_12460/3751647070.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  views_df['content_id'] = views_df['link'].str.split('/').str[-1].str.split('?').str[0]


,w/c,Channel ID,Channel Name,Channel URL,PlatformID,content_type,media,message,link,insights_engagements,insights_reach,duration,insights_video_views,insights_view_time,insights_avg_time_watched,insights_completion_rate,content_id
0,2025-08-04,TTK3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,Japan’s prime minister has described the count...,https://www.tiktok.com/@bbcnews/video/75369816...,49498,499763,51,604220,7914601,13,0.08,7536981698806811926
1,2025-08-04,TTK3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,A meteorite that crashed into a home in the US...,https://www.tiktok.com/@bbcnews/video/75369809...,1076289,9793095,14,11865744,115027940,10,0.24,7536980939285499158
2,2025-08-04,TTK3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,Are planes actually crashing more often? #Plan...,https://www.tiktok.com/@bbcnews/video/75369758...,54221,509953,251,595410,9325982,16,0.00,7536975832674372886
3,2025-08-04,TTK3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,Israeli Prime Minister Benjamin Netanyahu says...,https://www.tiktok.com/@bbcnews/video/75369647...,102208,2522087,18,3233846,45208070,14,0.30,7536964795443039510
4,2025-08-04,TTK3d596e6a-d239-434e-bb79-93e5d291b216,BBC News,https://www.tiktok.com/@bbcnews,TTK,post,video,"Zara, Next and Marks & Spencer have all had ad...",https://www.tiktok.com/@bbcnews/video/75369005...,73697,591508,67,785423,9696699,13,0.04,7536900507957349654


In [10]:


# optional: test video length is all in seconds
print(f"number of entries: {views_df.shape}")
views_df = views_df[~views_df['insights_reach'].isna()]
print(f"number of entries that have reach: {views_df.shape}")

cols_fill_nan = ['insights_avg_time_watched', 'duration', 'insights_reach',
                 'insights_completion_rate']
views_df[cols_fill_nan] = views_df[cols_fill_nan].fillna(0)  # or any other value you'd like


number of entries: (16558, 17)
number of entries that have reach: (16558, 17)


In [11]:
# Define x and y values for each row
views_df['x1'] = 0
views_df['x2'] = views_df['insights_avg_time_watched']
views_df['x3'] = views_df['duration']
views_df['x output'] = 10

views_df['y1'] = views_df['insights_reach']
views_df['y2'] = views_df['insights_reach'] / 2
views_df['y3'] = views_df['insights_reach'] * views_df['insights_completion_rate']

def apply_quadratic_fast(row):
    x_vals = np.array([row['x1'], row['x2'], row['x3']], dtype=float)
    y_vals = np.array([row['y1'], row['y2'], row['y3']], dtype=float)

    if len(set(x_vals)) < 3 or row['x3'] == 0:
        return 100

    # Build matrix and solve
    A = np.vstack([x_vals**2, x_vals, np.ones(3)]).T
    coeffs = np.linalg.solve(A, y_vals)

    # Evaluate at x output
    return coeffs[0]*row['x output']**2 + coeffs[1]*row['x output'] + coeffs[2]

# Create new column with interpolated values
views_df['30sec_video_views'] = views_df.apply(apply_quadratic_fast, axis=1).astype(float)
views_df['completed_video_views'] = views_df['insights_completion_rate'] * views_df['insights_reach']

conditions = [
    views_df['insights_reach'] == 0,
    (views_df['completed_video_views'].round(0) > views_df['30sec_video_views'].round(0)),
    views_df['insights_reach'] < views_df['30sec_video_views'],
    views_df['duration'] == 0
]

choices = [
    views_df['insights_engagements'],
    views_df['completed_video_views'],
    views_df['insights_reach'] * 0.799,
    views_df['insights_engagements']
]

views_df['final_video_views'] = np.select(conditions, choices, 
                                            default=views_df['30sec_video_views'])

In [12]:
views_df_full = views_df.merge(socialmedia_accounts[['Channel ID', 'ServiceID', 'Linked FB Account']],
               on='Channel ID', how='left')
test_functions.test_inner_join(views_df, socialmedia_accounts[['Channel ID', 'ServiceID', 'Linked FB Account']], 
                               ['Channel ID'], 
                               f"{platformID}_3_16", 
                               test_step='checking social media accounts in lookup, adding service',
                               focus='left')


✅ Inner join test TTK_3_16 successful: No issues found.
...updating logbook...



In [13]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['content_id', 'ServiceID', 'Channel ID', 'Channel Name', 'w/c', 
        'link',
        'final_video_views', 'Linked FB Account'
       ]
views_df_full = views_df_full[cols]
views_df_full.to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_views.csv",
                       index=None)


In [14]:
# YT views per viewer is missing for media action 
yt_views_per_viewer = pd.read_excel("../data/stale/YT views per viewer_TTKhelper.xlsx")[['w/c', 'Service', 'views_per_viewer']]
yt_views_per_viewer = yt_views_per_viewer.drop_duplicates(subset=['w/c', 'Service']).rename(columns={'Service': 'Lumen'})
yt_views_per_viewer = yt_views_per_viewer.merge(service_codes, on='Lumen', how='left').drop(columns='Lumen')
yt_views_per_viewer = gnl_expander(yt_views_per_viewer)
yt_views_per_viewer.sample()

,w/c,views_per_viewer,ServiceID
1838,2025-06-23,3.37,UKR


In [16]:
views_df_full[(views_df_full['w/c'] == '2025-03-31')
                & (views_df_full['link'] == 'https://www.tiktok.com/@bbcnewsswahili/video/7487892465597336838?utm_campaign=tt4d_open_api&utm_source=6997404317372137474')
]


,content_id,ServiceID,Channel ID,Channel Name,w/c,link,final_video_views,Linked FB Account
10975,7487892465597336838,SWA,TTK057b1686-d9f3-51fe-a129-5732678e609e,bbcnewsswahili,2025-03-31,https://www.tiktok.com/@bbcnewsswahili/video/7...,65.65,160894643929209


In [17]:
# to do: find out where duplicates happen here

views_df_full['w/c'] = pd.to_datetime(views_df_full['w/c'])
yt_views_per_viewer['w/c'] = pd.to_datetime(yt_views_per_viewer['w/c'])

views_df_yt = views_df_full.merge(yt_views_per_viewer, on=['ServiceID', 'w/c'], how='left', indicator=True)

matched = views_df_yt[views_df_yt['_merge'] == 'both'].drop(columns='_merge')
unmatched = views_df_yt[views_df_yt['_merge'] == 'left_only'].drop(columns=['_merge', 'views_per_viewer'])

views_per_viewer_by_service = yt_views_per_viewer.groupby(['ServiceID'])['views_per_viewer'].mean().reset_index()

matched_sec = unmatched.merge(views_per_viewer_by_service, on='ServiceID', how='left', indicator=True)
matched_sec = matched_sec[matched_sec['_merge'] == 'both'].drop(columns='_merge')

views_scaled = pd.concat([matched, matched_sec]).dropna(subset=['final_video_views'])
views_scaled = views_scaled[views_scaled['final_video_views'] > 0]
views_scaled['engaged_users'] = views_scaled['final_video_views']/(views_scaled['views_per_viewer']*1.14)

views_scaled.columns

Index(['content_id', 'ServiceID', 'Channel ID', 'Channel Name', 'w/c', 'link',
       'final_video_views', 'Linked FB Account', 'views_per_viewer',
       'engaged_users'],
      dtype='object')

In [18]:
views_scaled[(views_scaled['w/c'] == '2025-03-31')
                & (views_scaled['link'] == 'https://www.tiktok.com/@bbcnewsswahili/video/7487892465597336838?utm_campaign=tt4d_open_api&utm_source=6997404317372137474')
]


,content_id,ServiceID,Channel ID,Channel Name,w/c,link,final_video_views,Linked FB Account,views_per_viewer,engaged_users
10975,7487892465597336838,SWA,TTK057b1686-d9f3-51fe-a129-5732678e609e,bbcnewsswahili,2025-03-31,https://www.tiktok.com/@bbcnewsswahili/video/7...,65.65,160894643929209,2.08,27.68


# Country 

In [19]:
country_df = post_level_df.copy()

In [20]:

# Step 1: Parse the stringified list of country-percentage dictionaries
country_df['parsed_viewers'] = country_df['insights_viewers_by_country'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

# Step 2: Explode the parsed list into separate rows
exploded_df = country_df.explode('parsed_viewers').reset_index(drop=True)

# Step 3: Extract 'country' and 'percentage' from each dictionary
exploded_df[['country', 'percentage']] = exploded_df['parsed_viewers'].apply(
    lambda entry: pd.Series({
        'country': entry.get('country') if isinstance(entry, dict) else None,
        'percentage': entry.get('percentage') if isinstance(entry, dict) else None
    })
)

# Step 4: Drop the intermediate column
exploded_df.drop(columns=['parsed_viewers'], inplace=True)
exploded_df['country'] = exploded_df['country'].replace('Others', 'ZZ')
exploded_df['country'] = exploded_df['country'].fillna('ZZ')


In [21]:

exploded_df = exploded_df.rename(columns={'country': 'TikTok Codes'})
ttk_country_all = exploded_df.merge(country_codes[['TikTok Codes', 'PlaceID']], on='TikTok Codes', how='left',
                 indicator=True)

print(f"mismatches? \n{ttk_country_all._merge.value_counts()}")
ttk_country_all = ttk_country_all.drop(columns='_merge')

# Remove unknown countries (UNK)
ttk_country = ttk_country_all[ttk_country_all['PlaceID'] != 'UNK'].copy()

# Compute sum of percentages per video
sum_per_video = ttk_country.groupby('id')['percentage'].transform('sum')

# Rescale percentages
ttk_country['rescaled_percentage'] = ttk_country['percentage'] / sum_per_video

# Optional: Check that each video sums to 1
check = ttk_country.groupby('id')['rescaled_percentage'].sum()
check[~np.isclose(check, 1, atol=1e-6)]

country_cols = ['Channel ID', 'link', 'PlaceID', 'rescaled_percentage', 'w/c', 'PlatformID']
ttk_country= ttk_country[country_cols]
ttk_country.to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country.csv",
                  index=None, na_rep='')

mismatches? 
_merge
both          179560
left_only          0
right_only         0
Name: count, dtype: int64


In [22]:
# Weekly mean per channel/place to get to a rolling channel / coutnry split average
ttk_country_avg_channel = (
    ttk_country
    .groupby(['Channel ID', 'w/c', 'PlaceID', 'PlatformID'])['rescaled_percentage']
    .mean()
    .reset_index()
)

avg_country_df = calculate_rolling_avg_country_split(ttk_country_avg_channel, 'rescaled_percentage',
                                                     ttk_country['w/c'].min(), ttk_country['w/c'].max())



# combine views & country

country is a percentage by video. In the next section the actual metric is calculated (video views) per country. This is then summed to the profile level. 
As the average is larger than 1 the country view is rebased 


In [23]:
ttk_country[(ttk_country['w/c'] == '2025-03-31')
                & (ttk_country['PlaceID'] == 'TAN')
                & (ttk_country['link'] == 'https://www.tiktok.com/@bbcnewsswahili/video/7487892465597336838?utm_campaign=tt4d_open_api&utm_source=6997404317372137474')
]


,Channel ID,link,PlaceID,rescaled_percentage,w/c,PlatformID
118969,TTK057b1686-d9f3-51fe-a129-5732678e609e,https://www.tiktok.com/@bbcnewsswahili/video/7...,TAN,0.68,2025-03-31,TTK


In [24]:
views_scaled[(views_scaled['w/c'] == '2025-03-31')
                & (views_scaled['link'] == 'https://www.tiktok.com/@bbcnewsswahili/video/7487892465597336838?utm_campaign=tt4d_open_api&utm_source=6997404317372137474')
]


,content_id,ServiceID,Channel ID,Channel Name,w/c,link,final_video_views,Linked FB Account,views_per_viewer,engaged_users
10975,7487892465597336838,SWA,TTK057b1686-d9f3-51fe-a129-5732678e609e,bbcnewsswahili,2025-03-31,https://www.tiktok.com/@bbcnewsswahili/video/7...,65.65,160894643929209,2.08,27.68


In [75]:
# 1. Convert week column to datetime
ttk_country['w/c'] = pd.to_datetime(ttk_country['w/c'])

# 2. Remove merge indicators if present
ttk_country = ttk_country.drop(columns=['_merge'], errors='ignore')
views_scaled = views_scaled.drop(columns=['_merge'], errors='ignore')

# 3. Initial merge: country-level data with scaled views
merged_initial = ttk_country.merge(
    views_scaled,
    on=['Channel ID', 'link', 'w/c'],
    how='outer',
    indicator=True
)
print(f"Initial merge mismatches:\n{merged_initial._merge.value_counts()}")
matched_country = merged_initial[merged_initial['_merge'] == 'both'] 
matched_country['source'] = 'A'

# deal with country
unmatched_country = merged_initial[merged_initial['_merge'] == 'left_only']
# for each channel / w/c combination I want the country average of the previous 4 weeks to be the fill in for the missing data
merged_country_avg = avg_country_df[['w/c', 'Channel ID', 'PlaceID', 'rescaled_percentage']].merge(unmatched_country[views_scaled.columns],
    on=['Channel ID', 'w/c'],
    how='outer',
    indicator=True
)
print(f"Country second merge mismatches:\n{merged_country_avg._merge.value_counts()}")
# left only is fine these are weeky percentages in weeks that don't have uv data

# 4. Identify unmatched rows (right_only - country only)
unmatched_views = merged_initial[merged_initial['_merge'] == 'right_only']

# 5. Compute weekly averages for unmatched rows
weekly_avg = ttk_country.groupby(['Channel ID', 'w/c'])['rescaled_percentage'].mean().reset_index()
merged_weekly_avg = weekly_avg.merge(
    unmatched_views[views_scaled.columns],
    on=['Channel ID', 'w/c'],
    how='outer',
    indicator=True
)
print(f"Mismatches after weekly average merge:\n{merged_weekly_avg._merge.value_counts()}")

# 6. Identify remaining unmatched rows and compute channel-level averages
still_unmatched = merged_weekly_avg[merged_weekly_avg['_merge'] == 'right_only']
channel_avg = ttk_country.groupby(['Channel ID'])['rescaled_percentage'].mean().reset_index()
merged_channel_avg = channel_avg.merge(
    still_unmatched[views_scaled.columns],
    on=['Channel ID'],
    how='outer',
    indicator=True
)
print(f"Mismatches after channel-level average merge:\n{merged_channel_avg._merge.value_counts()}")
merged_country_avg['source'] = 'B'
merged_weekly_avg['source'] = 'C'
merged_channel_avg['source'] = 'D'
# 7. Combine all data sources (same as original logic)
combined_data = pd.concat(
    [matched_country, merged_country_avg, merged_weekly_avg, merged_channel_avg],
    ignore_index=True
).drop(columns=['_merge']).drop_duplicates()

# 8. Calculate country-level views at video level
combined_data['country_views_video_level'] = (
    combined_data['final_video_views'] * combined_data['rescaled_percentage']
)

# 9. add population column 
combined_data = combined_data.merge(country_codes[['PlaceID', gam_info['population_column']]],
                                    on=['PlaceID'], how='left')

combined_data.head()

Initial merge mismatches:
_merge
both          157039
left_only       1657
right_only       214
Name: count, dtype: int64
Country second merge mismatches:
_merge
both          70222
left_only     35958
right_only        0
Name: count, dtype: int64
Mismatches after weekly average merge:
_merge
left_only     497
both          214
right_only      0
Name: count, dtype: int64
Mismatches after channel-level average merge:
_merge
left_only     17
right_only     0
both           0
Name: count, dtype: int64


/var/folders/nk/x7c25nz92c1d6ljxz7l891cm0000gn/T/ipykernel_4385/1905420508.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matched_country['source'] = 'A'


,Channel ID,link,PlaceID,rescaled_percentage,w/c,PlatformID,content_id,ServiceID,Channel Name,final_video_views,Linked FB Account,views_per_viewer,engaged_users,source,country_views_video_level,Population2020
0,TTK057b1686-d9f3-51fe-a129-5732678e609e,https://www.tiktok.com/@bbcnewsswahili/video/7...,UK,0.02,2025-03-31,TTK,7487892465597336838,SWA,bbcnewsswahili,65.65,160894643929209,2.08,27.68,A,1.01,65045228
1,TTK057b1686-d9f3-51fe-a129-5732678e609e,https://www.tiktok.com/@bbcnewsswahili/video/7...,TAN,0.68,2025-03-31,TTK,7487892465597336838,SWA,bbcnewsswahili,65.65,160894643929209,2.08,27.68,A,44.49,23142960
2,TTK057b1686-d9f3-51fe-a129-5732678e609e,https://www.tiktok.com/@bbcnewsswahili/video/7...,UGA,0.01,2025-03-31,TTK,7487892465597336838,SWA,bbcnewsswahili,65.65,160894643929209,2.08,27.68,A,0.51,18502166
3,TTK057b1686-d9f3-51fe-a129-5732678e609e,https://www.tiktok.com/@bbcnewsswahili/video/7...,KEN,0.20,2025-03-31,TTK,7487892465597336838,SWA,bbcnewsswahili,65.65,160894643929209,2.08,27.68,A,12.93,46870422
4,TTK057b1686-d9f3-51fe-a129-5732678e609e,https://www.tiktok.com/@bbcnewsswahili/video/7...,ZAI,0.02,2025-03-31,TTK,7487892465597336838,SWA,bbcnewsswahili,65.65,160894643929209,2.08,27.68,A,1.37,16355917


In [27]:
combined_data[(combined_data['w/c'] == '2025-03-31')
                & (combined_data['PlaceID'] == 'TAN')
                #& (combined_data['link'] == 'https://www.tiktok.com/@bbcnewsswahili/video/7487892465597336838?utm_campaign=tt4d_open_api&utm_source=6997404317372137474')
]


,Channel ID,link,PlaceID,rescaled_percentage,w/c,PlatformID,content_id,ServiceID,Channel Name,final_video_views,Linked FB Account,views_per_viewer,engaged_users,source,Population2020,uv_by_country
1,TTK057b1686-d9f3-51fe-a129-5732678e609e,https://www.tiktok.com/@bbcnewsswahili/video/7...,TAN,0.68,2025-03-31,TTK,7487892465597336838,SWA,bbcnewsswahili,65.65,160894643929209,2.08,27.68,A,23142960,18.76


In [77]:
z

100%|████████████████████████████████████████████████████████████████| 17/17 [01:10<00:00,  4.17s/it]


In [28]:
dedupli_df[(dedupli_df['ServiceID'] == 'SWA') 
    & (dedupli_df['PlaceID'] == 'TAN')
    & (dedupli_df['w/c'] == '2025-03-31')
]

link,PlaceID,PlatformID,ServiceID,Channel ID,Population2020,w/c,uv_by_country
500,TAN,TTK,SWA,TTK057b1686-d9f3-51fe-a129-5732678e609e,23142960,2025-03-31,201379.85


In [79]:

# 12. Aggregate to global profile level
global_profile_views = dedupli_df.groupby(
    ['w/c', 'ServiceID', 'Channel ID']
)['country_views_channel_level'].sum().rename('global_views_channel_level').reset_index()

# 13. Merge country and global profile views
profile_views = dedupli_df.merge(
    global_profile_views,
    on=['ServiceID', 'w/c', 'Channel ID'],
    how='outer',
    indicator=True
)
print(f"Profile-level merge check:\n{profile_views._merge.value_counts()}")
profile_views = profile_views.drop(columns=['_merge'])

# 14. Calculate percentage contribution of country views
profile_views['profile_country_views_%'] = (
    profile_views['country_views_channel_level'] / profile_views['global_views_channel_level']
)

# 15. Merge back with scaled views for user-level metrics
ttk_df = profile_views.merge(
    views_scaled,
    on=['ServiceID', 'Channel ID', 'w/c'],
    how='inner'
)

# 16. Calculate UV by country
ttk_df['uv_by_country'] = (
    ttk_df['engaged_users'] * ttk_df['profile_country_views_%']
)


Profile-level merge check:
_merge
both          11954
left_only         0
right_only        0
Name: count, dtype: int64


In [30]:
print(dedupli_df.shape)
ttk_df = dedupli_df.dropna(subset='uv_by_country')
print(ttk_df.shape)

cols = ['w/c', 'PlaceID', 'ServiceID', 'Channel ID', 'uv_by_country', ]
ttk_df[cols].to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_uniqueViewer_country.csv", 
                     index=None)

(12301, 7)
(12301, 7)


In [ ]:
ttk_df[(ttk_df['ServiceID'] == 'SPA') 
    & (ttk_df['PlaceID'] == 'ARG')
    & (ttk_df['w/c'] == '2026-01-05')
].shape

In [ ]:
len(ttk_df[(ttk_df['ServiceID'] == 'SPA') 
    & (ttk_df['PlaceID'] == 'ARG')
    & (ttk_df['w/c'] == '2026-01-05')
]['link'].unique())